# M1A4 — Agente de Pesquisa com LangGraph + Tavily

Adaptação do notebook da aula com:
- **Dual provider** (Google Gemini 2.5-flash padrão ou OpenAI `gpt-4o-mini`) via `LLM_PROVIDER` no `.env`.
- SDK Tavily novo (`langchain-tavily`) — o `TavilySearchResults` do `langchain-community` foi descontinuado.
- Testes adicionais pedidos pela atividade (capital do Canadá, Copa 2014).

Padrão arquitetural: um grafo do LangGraph com dois nós (`llm` e `action`). Aresta condicional repete `llm → action → llm` enquanto o modelo emitir `tool_calls`; encerra quando responder texto puro.

## 1. Inicialização do ambiente

Carrega `.env`, escolhe provider, valida chaves (LLM + Tavily) e importa as libs do LangChain/LangGraph.

In [1]:
import os
import operator
from typing import Annotated, TypedDict
from dotenv import load_dotenv

load_dotenv()

PROVIDER = os.getenv("LLM_PROVIDER", "gemini").lower()
OPENAI_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash"

assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY ausente no .env (https://tavily.com)"
if PROVIDER == "openai":
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY ausente no .env"
elif PROVIDER == "gemini":
    assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY ausente no .env"
else:
    raise ValueError(f"LLM_PROVIDER desconhecido: {PROVIDER}")

from langgraph.graph import StateGraph, END
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_tavily import TavilySearch

print(f"Provider ativo: {PROVIDER}")

Provider ativo: gemini


## 2. Ferramenta de busca (Tavily)

`TavilySearch` lê `TAVILY_API_KEY` automaticamente do ambiente. `max_results=3` limita o ruído de cada busca.

In [2]:
tool = TavilySearch(max_results=3)

## 3. Estado compartilhado entre nós do grafo

`AgentState` mantém o histórico de mensagens. O `Annotated[..., operator.add]` ensina o LangGraph a **concatenar** mensagens novas em cada atualização (em vez de sobrescrever).

In [3]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

## 4. Classe `Agent` (grafo LangGraph)

Dois nós:
1. **`llm`** — chama o modelo (`call_model`) com o histórico atual.
2. **`action`** — executa cada `tool_call` que o modelo emitiu (`take_action`).

A aresta condicional vai pra `action` se houver `tool_calls` pendentes, senão encerra (`END`). Depois de `action`, sempre volta pra `llm`.

In [4]:
class Agent:
    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_model)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END},
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState) -> bool:
        last = state["messages"][-1]
        return bool(getattr(last, "tool_calls", None))

    def call_model(self, state: AgentState):
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {"messages": [message]}

    def take_action(self, state: AgentState):
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Chamando: {t}")
            if t["name"] not in self.tools:
                print("  -- nome de ferramenta incorreto --")
                result = "nome de ferramenta incorreto, tente novamente"
            else:
                result = self.tools[t["name"]].invoke(t["args"])
            results.append(ToolMessage(tool_call_id=t["id"], name=t["name"], content=str(result)))
        print("De volta ao modelo!")
        return {"messages": results}

## 5. Prompt de sistema (assistente de pesquisa)

Orienta o LLM a ser um pesquisador disciplinado: usar o motor de busca quando precisar de fatos atualizados, encadear múltiplas buscas se necessário.

In [5]:
prompt = (
    "Você é um assistente de pesquisa inteligente. Use o motor de busca para procurar informações. "
    "Você pode fazer múltiplas chamadas (juntas ou em sequência). "
    "Só procure informações quando tiver certeza do que quer. "
    "Se precisar procurar algumas informações antes de fazer uma pergunta de acompanhamento, você pode fazer isso!"
)

## 6. Instanciando o modelo com ferramentas

Escolhe `ChatGoogleGenerativeAI` ou `ChatOpenAI` conforme o provider. Ambos suportam **tool calling** nativo — o LangGraph cuida do roteamento.

In [6]:
if PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    model = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
else:
    from langchain_google_genai import ChatGoogleGenerativeAI
    model = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)

abot = Agent(model, [tool], system=prompt)
print(f"Modelo: {model.__class__.__name__}")

Modelo: ChatGoogleGenerativeAI


## 7. Teste 1 — Pergunta única (1 chamada de ferramenta)

Clima no Rio: força uma busca atualizada e responde com o que veio do Tavily.

In [7]:
messages = [HumanMessage(content="Qual é o clima no Rio de Janeiro?")]
result = abot.graph.invoke({"messages": messages})
print("\nResposta final:")
print(result["messages"][-1].content)

Chamando: {'name': 'tavily_search', 'args': {'query': 'clima Rio de Janeiro'}, 'id': '9bea6e3c-9c7e-4a2a-841c-c36eb08367fd', 'type': 'tool_call'}
De volta ao modelo!

Resposta final:
[{'type': 'text', 'text': 'No Rio de Janeiro, a sensação térmica é de 22°C, com ventos de 9 km/h e 73% de nebulosidade. A radiação UV está moderada. Não há previsão de chuva para os próximos 15 dias.', 'extras': {'signature': 'CswCAQw51sftVIhU6msPoigRxENV3Rk5y0kWtQO/BmTVNJBm82+f64LOA+Sk8J7EmJPqCDAPI/trhW3oJyljdjaW5UXCDq5Pv3cn4uY1dfyTNRtQsOGauOmBcwhGXi6AB0XBA5uOmhaewhIGZjdFE0hDwTt0c8uCpKYArFyuEuRnPx9y3c3+z8dyGHtMbNety2KtZ+2xhpAHn5z//GIDdZANPwiiPyldsl9tuBGenitF6lkYLS6H+Sve+uCIFobYtwtLhyYLY2jjpoIDrloZcfWoPsPlNNCYPKNLbIDvtKSJT28OX6y/WphErfhd2OKCRJQwBLzATSY8LoJYK3HBwuMNeZdk54M4XdJeOtmtV1LikpkyWLrpl5XzBKSN5sU1vOueWkK4uLNDISXAH+Aqg9riYN9NRQ73i2nxMih2+dP83e6epKUNYBEgBawPNyE='}}]


## 8. Teste 2 — Capital do Canadá

Pergunta factual que o LLM provavelmente já sabe, mas o prompt manda buscar se quiser confirmar. Observe se o agente faz busca ou responde direto.

In [8]:
abot = Agent(model, [tool], system=prompt)
result = abot.graph.invoke({"messages": [HumanMessage(content="Qual a capital do Canadá?")]})
print("\nResposta final:")
print(result["messages"][-1].content)

Chamando: {'name': 'tavily_search', 'args': {'query': 'capital do Canadá'}, 'id': '3ab2de16-9626-42a9-a957-19d0e7ad8ebc', 'type': 'tool_call'}
De volta ao modelo!

Resposta final:
[{'type': 'text', 'text': 'A capital do Canadá é Ottawa.', 'extras': {'signature': 'CocCAQw51seEQvDGNMerIAb4aXdA1QWy+oeMmCyjbhWDBXSs6F37FWkveoD/dPHpx+aBY6JLpaHPMKvFRBKeEscnJmKTaCgMuu4w/PK8jlTyS/9YQRurEFXD8cGkaPCmF7KMeGLjCsQ5/Btz8hpNC6sVX1R6LMOwarFGbZEa2HB679i/ogOhF09NAYX0QRvoqyxQjE3R4zKqrWNwnsTfPBuRRZ7Sp7GNC8gVKZN2t9LumkEzrYbQPzh6Ctm5P5ic56sOyGH8kI/9i5vxr/RJ+syGawwDx07H46goRuRzXIU4Eu0SWZqZSz4NB7PXKygf5npRfLP8B+5xoj7fasIZCkj5g120h7cDslg='}}]


## 9. Teste 3 — Copa do Mundo de 2014

Pergunta factual datada — espera-se 1 busca + resposta direta.

In [9]:
abot = Agent(model, [tool], system=prompt)
result = abot.graph.invoke({"messages": [HumanMessage(content="Quem ganhou a Copa do Mundo de 2014?")]})
print("\nResposta final:")
print(result["messages"][-1].content)

Chamando: {'name': 'tavily_search', 'args': {'query': 'Quem ganhou a Copa do Mundo de 2014?'}, 'id': '3e393075-1b28-40b2-bdd3-2b67dd655528', 'type': 'tool_call'}
De volta ao modelo!

Resposta final:
[{'type': 'text', 'text': 'A Alemanha ganhou a Copa do Mundo de 2014, vencendo a Argentina na final.', 'extras': {'signature': 'CsIBAQw51sd1TpMy3jmv7LxMcIli9Xx1ISimasz16iuN9tRuQAjI9HiZ09bFI3M+UheAy3PN4tV5Xg5q/g1iDsWNrSM5l8RpjG3D5NFB6L64W78KQDs1rlGV+f8e5PJ5WsqXNsHQqZOQkqq55vgxGEPkCOv1kiJ3THuXIVRxS9LmRg6CYKX2HIs5cP5j/50htHo8BSV4/ncD/wYQ1Iy+p4lGb6SUNBycZhAftRelH19Ba/XtfFYd+JoiYV/aoTbzUNhFBF0='}}]


## 10. Teste 4 — Pergunta composta (múltiplas buscas)

Reproduz o exemplo do notebook original: campeão da Copa 2022 + PIB do país. O grafo permite duas `tool_calls` em sequência (ou paralelo, dependendo do modelo).

In [10]:
query = "Quem ganhou a Copa do Mundo de 2022? Qual é o PIB desse país? Responda cada pergunta."
abot = Agent(model, [tool], system=prompt)
result = abot.graph.invoke({"messages": [HumanMessage(content=query)]})
print("\nResposta final:")
print(result["messages"][-1].content)

Chamando: {'name': 'tavily_search', 'args': {'query': 'quem ganhou a Copa do Mundo de 2022?'}, 'id': '78c19b1d-22b7-467f-b198-789f6429051f', 'type': 'tool_call'}
De volta ao modelo!
Chamando: {'name': 'tavily_search', 'args': {'query': 'PIB da Argentina'}, 'id': 'f6263c1a-24d4-4e8f-81b2-0179964e34f1', 'type': 'tool_call'}
De volta ao modelo!

Resposta final:
[{'type': 'text', 'text': 'A Argentina ganhou a Copa do Mundo de 2022.\n\nO Produto Interno Bruto (PIB) da Argentina foi de 633,27 bilhões de dólares americanos em 2024, de acordo com dados oficiais do Banco Mundial. O pico foi de 646,08 bilhões de dólares americanos em 2023.', 'extras': {'signature': 'CoUCAQw51selC0JE6CwCLuE4RupP2eqnex4xLfthZVjMfkCZ9H30wCnPHS1F1K8KKk/pKTKtcwLcJjjVyYBtA1ie7r8oViJ5lhEeB3V8bEqz4WNaWZlRYEXog44Xh20M96SiDAlmQWb0VQVicylVaRWAoR/OwakZ2gpBG0vNpfLrfIKiyKxb6XJHaIcWTNqX1PUES1shHXsPr1BFredlAmG9avZbbEwGeGRu+OMo6V0XqHwQBUNu+Wk3HyLqvrhGZkDba8B+7c0ZmN26WvfSE6GhbQnof4LGnAD1R6NvJZITQgeUw3hSceCB38+kgraXBYgSiEQpNqPUbZL

## 11. Teste 5 — Pergunta composta com pegadinha geográfica

Força o agente a fazer **duas buscas distintas** e cruzar os resultados pra responder se as duas cidades coincidem. A pegadinha: muita gente assume que a maior cidade é a capital — não é o caso da Austrália (capital = Canberra; maior cidade = Sydney).

In [ ]:
query = (
    "Qual a capital da Austrália? E qual a maior cidade do país? "
    "São a mesma cidade? Explique."
)
abot = Agent(model, [tool], system=prompt)
result = abot.graph.invoke({"messages": [HumanMessage(content=query)]})
print("\nResposta final:")
print(result["messages"][-1].content)